**Week 4 — Eigenvalue, SVD, Low-rank approximation และ PCA (ชุดฝึกธุรกิจ)**

### สรุปเงื่อนไขการประมวลผลข้อมูล
- **การอ่านไฟล์:**
  - ข้อมูลลูกค้า: `pd.read_csv('biz_customers.csv')` (1203 แถว)
  - ข้อมูลแคมเปญ: `pd.read_csv('biz_campaigns.csv')` (240 แถว)
  - ข้อมูลเมทริกซ์ A: `pd.read_csv('pb04_A.csv')` (ขนาด 4×4)
  - ข้อมูลยอดขายสาขา: `pd.read_csv('pb04_sales.csv')` (ขนาด 16×12)
- **การนับดัชนี (0-based Indexing):** เริ่มนับจาก 0 ทั้งแถวและคอลัมน์
- **PCn (Principal Component):** เรียงตาม eigenvalue จากมากไปน้อย (PC1 = ตัวใหญ่สุด)
- **Loading:** คือสมาชิกของ eigenvector ที่มีความยาว 1 ของ PC นั้น
- **PC Score:** คือพิกัดของแถวข้อมูลหลังฉายลงบนแกน PC นั้น (เช่นค่าจาก `pca.transform()`)
- **Standardization:** $z = (x - \text{mean})/\text{std}$ โดยใช้ $\text{std}$ แบบ `ddof=0` (เช่น `StandardScaler`)

### รายละเอียดไฟล์ข้อมูล

**ไฟล์ลูกค้า `biz_customers.csv`** (1203 แถว)
- `age`: อายุของลูกค้า (ปี)
- `income_k`: รายได้ต่อเดือน (พันบาท)
- `tenure_months`: ระยะเวลาที่เป็นลูกค้า (เดือน)
- `channel`: ช่องทางที่ซื้อบ่อย (`online`, `store`)
- `loyalty_member`: สมาชิกสะสมแต้ม (1 = ใช่, 0 = ไม่ใช่)
- `web_visits`: จำนวนครั้งที่เข้าเว็บ/แอปใน 30 วัน
- `email_opens`: จำนวนอีเมลโปรโมชันที่เปิดอ่าน (จาก 20 ฉบับ)
- `basket_size`: จำนวนสินค้าเฉลี่ยต่อออเดอร์ (ขนาดตะกร้า)
- `spend_k`: ยอดซื้อต่อเดือน (พันบาท)
- `orders_90d`: จำนวนออเดอร์ใน 90 วัน

**ไฟล์แคมเปญ `biz_campaigns.csv`** (240 แถว)
- `tv`: งบโฆษณาทางโทรทัศน์ (พันบาท)
- `radio`: งบโฆษณาทางวิทยุ (พันบาท)
- `social`: งบโฆษณาทางโซเชียลมีเดีย (พันบาท)
- `email`: งบโฆษณาทางอีเมล (พันบาท)
- `season`: ไตรมาสที่จัดแคมเปญ (`Q1`–`Q4`)
- `sales`: ยอดขายของแคมเปญ (พันชิ้น)
- `total_budget`: งบโฆษณารวมทุกสื่อ (พันบาท)

**ไฟล์เมทริกซ์ `pb04_A.csv`** (ขนาด 4×4)
- เมทริกซ์ $A$ บอกผลกระทบข้ามกันของยอดขายระหว่าง 4 กลุ่มสินค้า

**ไฟล์ยอดขาย `pb04_sales.csv`** (ขนาด 16×12)
- ยอดขายรายเดือน (พันบาท) ของ 16 สาขา (แถว) ตลอด 12 เดือน (คอลัมน์)

**โจทย์ข้อ 4.1:** ไฟล์ `pb04_A.csv` คือเมทริกซ์ $A$ ขนาด $4 \times 4$ ที่บอกผลกระทบข้ามกันของยอดขายระหว่าง 4 กลุ่มสินค้า (ไม่ใช่เมทริกซ์สมมาตร และมี eigenvalue เป็นจำนวนจริงทั้ง 4 ตัว) eigenvalue ที่ **ใหญ่เป็นอันดับสอง** (เรียงตามค่าจากมากไปน้อย ไม่ใช่ตามค่าสัมบูรณ์) = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านไฟล์ pb04_A.csv (ขนาด 4x4)
A = pd.read_csv('pb04_A.csv').to_numpy(dtype=np.float64)
print(f"ขนาดของ A: {A.shape}")

# 2. คำนวณ eigenvalues
eigvals = np.linalg.eigvals(A)

# 3. เรียงตามค่าจริงจากมากไปน้อย (descending order)
eigvals_sorted = np.sort(np.real(eigvals))[::-1]
print(f"Eigenvalues ทั้งหมด (เรียงจากมากไปน้อย): {eigvals_sorted}")

# 4. เลือกตัวที่ใหญ่เป็นอันดับสอง (index 1)
ans_41 = eigvals_sorted[1]
print(f"คำตอบข้อ 4.1: {ans_41:.4f}")


ขนาดของ A: (4, 4)
Eigenvalues ทั้งหมด (เรียงจากมากไปน้อย): [ 7.43461727 -0.80106334 -2.41157397 -5.22197995]
คำตอบข้อ 4.1: -0.8011


**โจทย์ข้อ 4.2:** Standardize ข้อมูล 4 คอลัมน์ `age`, `income_k`, `basket_size`, `orders_90d` ของลูกค้าทั้ง 1203 คนใน `biz_customers.csv` ด้วย $z = (x - \text{mean})/\text{std}$ แล้วทำ PCA สัดส่วนความแปรปรวนที่ PC1 อธิบายได้ = ? (ทศนิยม 4 ตำแหน่ง; ตอบเป็นสัดส่วน 0–1 ไม่ใช่เปอร์เซ็นต์)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1. อ่านข้อมูล 4 คอลัมน์ของลูกค้า 1203 คน
df = pd.read_csv('biz_customers.csv')
cols = ['age', 'income_k', 'basket_size', 'orders_90d']
X = df[cols].values

# 2. Standardize ด้วย z = (x - mean)/std (ddof=0)
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

# 3. ทำ PCA
pca = PCA()
pca.fit(X_std)

# 4. ดูสัดส่วนความแปรปรวนที่อธิบายได้ (explained variance ratio)
var_ratio = pca.explained_variance_ratio_
print(f"สัดส่วนความแปรปรวนแต่ละ PC: {var_ratio}")
ans_42 = var_ratio[0]
print(f"คำตอบข้อ 4.2: {ans_42:.4f}")


สัดส่วนความแปรปรวนแต่ละ PC: [0.3689426  0.26769933 0.22601601 0.13734206]
คำตอบข้อ 4.2: 0.3689


**โจทย์ข้อ 4.3:** กรณีศึกษาทีมข้อมูลของแพลตฟอร์มอีคอมเมิร์ซมีข้อมูล 8 ตัวแปร ได้แก่อายุ, รายได้ต่อเดือน, ระยะเวลาที่เป็นลูกค้า, จำนวนครั้งที่เข้าเว็บ/แอป, จำนวนอีเมลที่เปิดอ่าน, ขนาดตะกร้า, ยอดซื้อต่อเดือน, จำนวนออเดอร์ใน 90 วัน (`age`, `income_k`, `tenure_months`, `web_visits`, `email_opens`, `basket_size`, `spend_k`, `orders_90d`) ของลูกค้า 1203 คน และต้องการลดมิติเพื่อใช้เป็นอินพุตของโมเดลทำนายการเลิกซื้อ จึง standardize ข้อมูลทุกตัวแปรแล้วทำ PCA เลือกจำนวน principal component น้อยที่สุดที่ทำให้ **ความแปรปรวนสะสมที่อธิบายได้ $\ge 60\%$** (cumulative explained variance ratio $\ge 0.60$) ต้องใช้ PC อย่างน้อย ? ตัว (จำนวนเต็ม)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1. อ่านข้อมูล 8 ตัวแปร
df = pd.read_csv('biz_customers.csv')
cols = ['age', 'income_k', 'tenure_months', 'web_visits', 'email_opens', 'basket_size', 'spend_k', 'orders_90d']
X = df[cols].values

# 2. Standardize ทุกตัวแปร
X_std = StandardScaler().fit_transform(X)

# 3. ทำ PCA และหาผลรวมสะสมของสัดส่วนความแปรปรวน (cumulative explained variance)
pca = PCA().fit(X_std)
cum_var = np.cumsum(pca.explained_variance_ratio_)

for i, cv in enumerate(cum_var):
    print(f"PC1 ถึง PC{i+1}: ความแปรปรวนสะสม = {cv:.4f} ({cv*100:.2f}%)")

# 4. หาจำนวน PC น้อยที่สุดที่ทำให้ผลรวมสะสม >= 0.60
ans_43 = int(np.argmax(cum_var >= 0.60) + 1)
print(f"คำตอบข้อ 4.3: {ans_43}")


PC1 ถึง PC1: ความแปรปรวนสะสม = 0.2397 (23.97%)
PC1 ถึง PC2: ความแปรปรวนสะสม = 0.4459 (44.59%)
PC1 ถึง PC3: ความแปรปรวนสะสม = 0.5752 (57.52%)
PC1 ถึง PC4: ความแปรปรวนสะสม = 0.6904 (69.04%)
PC1 ถึง PC5: ความแปรปรวนสะสม = 0.7955 (79.55%)
PC1 ถึง PC6: ความแปรปรวนสะสม = 0.8825 (88.25%)
PC1 ถึง PC7: ความแปรปรวนสะสม = 0.9500 (95.00%)
PC1 ถึง PC8: ความแปรปรวนสะสม = 1.0000 (100.00%)
คำตอบข้อ 4.3: 4


**โจทย์ข้อ 4.4:** กรณีศึกษานักวิเคราะห์ใหม่ของเครือร้านค้าทำ PCA กับ 5 ตัวแปร ได้แก่งบโฆษณาทีวี, งบโฆษณาวิทยุ, งบโฆษณาโซเชียล, งบโฆษณาอีเมล, ยอดขาย (`tv`, `radio`, `social`, `email`, `sales`) ของแคมเปญโฆษณาทั้ง 240 รายการ โดย**ใช้ค่าดิบ ไม่ standardize** (หักค่าเฉลี่ยเท่านั้น เช่นที่ `sklearn.decomposition.PCA` ทำให้อัตโนมัติ) สัดส่วนความแปรปรวนที่ PC1 อธิบายได้ = ? (ทศนิยม 4 ตำแหน่ง; ตอบเป็นสัดส่วน 0–1)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

# 1. อ่านข้อมูล 5 ตัวแปรของแคมเปญ 240 รายการ
df = pd.read_csv('biz_campaigns.csv')
cols = ['tv', 'radio', 'social', 'email', 'sales']
X = df[cols].values

# 2. ทำ PCA บนค่าดิบ (sklearn PCA จะ mean-center อัตโนมัติ แต่ไม่ scale variance)
pca = PCA()
pca.fit(X)

# 3. สัดส่วนความแปรปรวนของ PC1
var_ratio = pca.explained_variance_ratio_
print(f"สัดส่วนความแปรปรวนแต่ละ PC (ค่าดิบ): {var_ratio}")
ans_44 = var_ratio[0]
print(f"คำตอบข้อ 4.4: {ans_44:.4f}")


สัดส่วนความแปรปรวนแต่ละ PC (ค่าดิบ): [9.24630641e-01 3.93055369e-02 2.72821150e-02 8.31492176e-03 4.66785313e-04]
คำตอบข้อ 4.4: 0.9246


**โจทย์ข้อ 4.5:** ฝึกสร้างเมทริกซ์ $Z$ ขนาด $1203 \times 5$ จากลูกค้าทั้ง 1203 คนใน `biz_customers.csv` คอลัมน์ `age`, `income_k`, `email_opens`, `basket_size`, `orders_90d` (ตามลำดับนี้) โดย standardize แต่ละคอลัมน์ $z = (x - \text{mean})/\text{std}$ ใช้ $\text{std}$ แบบ `ddof=0` (เหมือน `np.std` ค่าเริ่มต้น) แล้วทำ SVD หา rank-2 approximation $Z_2$ ที่ดีที่สุด ค่า Frobenius norm $\|Z - Z_2\|_F$ = ? (ทศนิยม 3 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. อ่านข้อมูล 5 คอลัมน์
df = pd.read_csv('biz_customers.csv')
cols = ['age', 'income_k', 'email_opens', 'basket_size', 'orders_90d']
X = df[cols].values

# 2. Standardize ddof=0
Z = StandardScaler().fit_transform(X)
print(f"ขนาดเมทริกซ์ Z: {Z.shape}")

# 3. คำนวณ SVD: Z = U @ diag(s) @ Vt
U, s, Vt = np.linalg.svd(Z, full_matrices=False)
print(f"Singular values ทั้ง 5 ตัว: {s}")

# 4. คำนวณ Z_2 (rank-2 approximation ที่ดีที่สุดตาม Eckart-Young Theorem)
# และหา Frobenius norm ||Z - Z_2||_F
Z_2 = U[:, :2] @ np.diag(s[:2]) @ Vt[:2, :]
ans_45 = np.linalg.norm(Z - Z_2, 'fro')
print(f"Frobenius norm โดยตรง: {ans_45:.3f}")
# หรือคำนวณจากสูตร: sqrt(sum(s_i^2 สำหรับ i >= 2))
formula_norm = np.sqrt(np.sum(s[2:]**2))
print(f"Frobenius norm จาก singular values ที่ตัดทิ้ง: {formula_norm:.3f}")
print(f"คำตอบข้อ 4.5: {ans_45:.3f}")


ขนาดเมทริกซ์ Z: (1203, 5)
Singular values ทั้ง 5 ตัว: [42.2827452  38.98447936 33.00915142 31.06413246 25.54986153]
Frobenius norm โดยตรง: 52.032
Frobenius norm จาก singular values ที่ตัดทิ้ง: 52.032
คำตอบข้อ 4.5: 52.032


**โจทย์ข้อ 4.6:** กรณีศึกษาทีมวิจัยตลาดต้องการยุบตัวแปร 3 ตัว ได้แก่งบโฆษณาวิทยุ, งบโฆษณาอีเมล, ยอดขาย (`radio`, `email`, `sales`) ของแคมเปญโฆษณาทั้ง 240 รายการเหลือดัชนีเดียวตามทิศทางที่ข้อมูลกระจายมากที่สุด จึงหา covariance matrix $3 \times 3$ ของสามตัวแปรนี้ (ไม่ standardize ใช้ `np.cov` แบบ $n-1$) แล้วหา eigenvector (ความยาว 1) ของ eigenvalue ที่ใหญ่ที่สุด ค่าสัมบูรณ์ของ loading ของยอดขายใน PC1 นี้ = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านข้อมูล 3 ตัวแปร
df = pd.read_csv('biz_campaigns.csv')
cols = ['radio', 'email', 'sales']
X = df[cols].values

# 2. หา covariance matrix 3x3 (ddof=1 / หาร n-1)
cov_matrix = np.cov(X, rowvar=False)
print("Covariance matrix (3x3):\n", cov_matrix)

# 3. คำนวณ eigenvalues และ eigenvectors
eigvals, eigvecs = np.linalg.eig(cov_matrix)
print(f"Eigenvalues: {eigvals}")

# 4. หา eigenvector ของ eigenvalue ที่ใหญ่ที่สุด (PC1)
idx_max = np.argmax(eigvals)
pc1_vec = eigvecs[:, idx_max]
print(f"PC1 eigenvector: {pc1_vec}")

# 5. ค่าสัมบูรณ์ของ loading ของ 'sales' (ตัวแปรลำดับที่ 3, index 2)
ans_46 = np.abs(pc1_vec[2])
print(f"คำตอบข้อ 4.6: {ans_46:.4f}")


Covariance matrix (3x3):
 [[201.76189568   4.10238634  79.23194098]
 [  4.10238634  40.40795415   2.39294979]
 [ 79.23194098   2.39294979  91.01844991]]
Eigenvalues: [223.75604147  39.56105869  69.8711999 ]
PC1 eigenvector: [-0.91883405  0.05344187 -0.39100889]
คำตอบข้อ 4.6: 0.3910


**โจทย์ข้อ 4.7:** Standardize ข้อมูล 4 คอลัมน์ `age`, `income_k`, `web_visits`, `email_opens` ของลูกค้าทั้ง 1203 คนใน `biz_customers.csv` ด้วย $z = (x - \text{mean})/\text{std}$ โดยใช้ $\text{std}$ แบบ `ddof=0` (เหมือน `StandardScaler`) แล้วทำ PCA คะแนนบนแกน PC1 ของลูกค้าแถวที่ 466 (พิกัดของแถวนั้นหลังฉายลงบน PC1 เช่นค่าจาก `transform` ของ sklearn) ตอบเป็นค่าสัมบูรณ์ $|\text{คะแนน}|$ = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1. อ่านข้อมูล 4 ตัวแปร
df = pd.read_csv('biz_customers.csv')
cols = ['age', 'income_k', 'web_visits', 'email_opens']
X = df[cols].values

# 2. Standardize ddof=0
X_std = StandardScaler().fit_transform(X)

# 3. ทำ PCA และหาคะแนน (transform ข้อมูลลงบนแกน PC)
pca = PCA()
scores = pca.fit_transform(X_std)

# 4. หาคะแนนบนแกน PC1 ของแถวที่ 466 (0-indexed)
score_row466_pc1 = scores[466, 0]
ans_47 = np.abs(score_row466_pc1)
print(f"คะแนน PC1 ของแถวที่ 466: {score_row466_pc1}")
print(f"คำตอบข้อ 4.7: {ans_47:.4f}")


คะแนน PC1 ของแถวที่ 466: 0.9934762332020566
คำตอบข้อ 4.7: 0.9935


**โจทย์ข้อ 4.8:** ไฟล์ `pb04_sales.csv` คือยอดขายรายเดือน (พันบาท) ของ 16 สาขา (แถว) ตลอด 12 เดือน (คอลัมน์) เป็นเมทริกซ์ $A$ ขนาด $16 \times 12$ (ใช้ค่าดิบ ไม่ต้องหักค่าเฉลี่ยหรือ standardize) ทำ SVD แล้วสร้างเมทริกซ์ประมาณ rank-3 ที่ดีที่สุด $A_3 = \sum_{i=1}^3 \sigma_i u_i v_i^T$ ค่าที่แถว 8 คอลัมน์ 7 ของ $A_3$ (นับจาก 0 ทั้งแถวและคอลัมน์) = ? (ทศนิยม 2 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านไฟล์ pb04_sales.csv (ขนาด 16x12)
A = pd.read_csv('pb04_sales.csv').to_numpy(dtype=np.float64)
print(f"ขนาดของเมทริกซ์ A: {A.shape}")

# 2. ทำ SVD บนค่าดิบ
U, s, Vt = np.linalg.svd(A, full_matrices=False)

# 3. สร้างเมทริกซ์ประมาณ rank-3 ที่ดีที่สุด (A_3)
A_3 = U[:, :3] @ np.diag(s[:3]) @ Vt[:3, :]

# 4. อ่านค่าที่แถว 8 คอลัมน์ 7 (0-indexed)
ans_48 = A_3[8, 7]
print(f"ค่าที่แถว 8 คอลัมน์ 7 ของ A_3: {ans_48:.2f}")
print(f"คำตอบข้อ 4.8: {ans_48:.2f}")


ขนาดของเมทริกซ์ A: (16, 12)
ค่าที่แถว 8 คอลัมน์ 7 ของ A_3: 63.50
คำตอบข้อ 4.8: 63.50


**โจทย์ข้อ 4.9:** กรณีศึกษาศูนย์กระจายสินค้าต้องส่งตารางยอดขายรายเดือนของ 16 สาขาตลอด 12 เดือนในไฟล์ `pb04_sales.csv` ผ่านเครือข่ายที่ช้า จึงเก็บเฉพาะ singular value 4 ตัวแรก (รวม singular vector ที่คู่กัน) ทีมวัดคุณภาพด้วยความคลาดเคลื่อนสัมพัทธ์ $\|A - A_4\|_F / \|A\|_F$ ($A$ คือเมทริกซ์ยอดขายดิบ ไม่หักค่าเฉลี่ย; $A_4$ คือเมทริกซ์ประมาณ rank-4 ที่ดีที่สุด) ค่านี้ = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านข้อมูลยอดขาย 16x12
A = pd.read_csv('pb04_sales.csv').to_numpy(dtype=np.float64)

# 2. ทำ SVD บนค่าดิบ
U, s, Vt = np.linalg.svd(A, full_matrices=False)

# 3. สร้างเมทริกซ์ประมาณ rank-4 ที่ดีที่สุด A_4
A_4 = U[:, :4] @ np.diag(s[:4]) @ Vt[:4, :]

# 4. คำนวณความคลาดเคลื่อนสัมพัทธ์ Frobenius norm
norm_diff = np.linalg.norm(A - A_4, 'fro')
norm_A = np.linalg.norm(A, 'fro')
ans_49 = norm_diff / norm_A
print(f"ความคลาดเคลื่อนสัมพัทธ์ (คำนวณโดยตรง): {ans_49:.4f}")

# หรือคำนวณผ่าน singular values: sqrt(sum(s[4:]^2)) / sqrt(sum(s^2))
formula_rel = np.sqrt(np.sum(s[4:]**2)) / np.sqrt(np.sum(s**2))
print(f"ความคลาดเคลื่อนสัมพัทธ์ (จากสูตร singular values): {formula_rel:.4f}")
print(f"คำตอบข้อ 4.9: {ans_49:.4f}")


ความคลาดเคลื่อนสัมพัทธ์ (คำนวณโดยตรง): 0.0434
ความคลาดเคลื่อนสัมพัทธ์ (จากสูตร singular values): 0.0434
คำตอบข้อ 4.9: 0.0434


**โจทย์ข้อ 4.10:** กรณีศึกษาธนาคารที่ออกบัตรเครดิตร่วมกับร้านค้าสร้างดัชนีลดมิติจาก 5 ตัวแปร ได้แก่อายุ, รายได้ต่อเดือน, ระยะเวลาที่เป็นลูกค้า, ขนาดตะกร้า, ยอดซื้อต่อเดือน (`age`, `income_k`, `tenure_months`, `basket_size`, `spend_k`) โดยใช้เฉพาะ**ลูกค้า 960 คนแรก**ของไฟล์เป็นข้อมูลสำหรับเรียนรู้ ขั้นแรก standardize ด้วย mean และ std แบบ ddof=0 ที่คำนวณจาก 960 คนแรกนี้ (เหมือน `StandardScaler().fit` บนข้อมูลชุดนั้น) แล้วทำ PCA บนข้อมูลชุดเดียวกัน จากนั้นนำ**ลูกค้า 243 คนที่เหลือ** (แถวที่ 960 ถึง 1202) มา standardize ด้วย mean/std ชุดเดิม (ไม่คำนวณใหม่) แล้วฉายลงบน PC1 ความแปรปรวนตัวอย่าง (หาร $n-1$ เหมือน `.var(ddof=1)` ของ pandas) ของคะแนน PC1 ของ 243 คนนี้ = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1. อ่านข้อมูลลูกค้า 5 ตัวแปร
df = pd.read_csv('biz_customers.csv')
cols = ['age', 'income_k', 'tenure_months', 'basket_size', 'spend_k']

# 2. แบ่งข้อมูล: 960 คนแรก (Train) และ 243 คนที่เหลือ (Test: แถว 960 ถึง 1202)
train_df = df.iloc[:960][cols]
test_df = df.iloc[960:][cols]
print(f"ขนาด Train: {train_df.shape}, ขนาด Test: {test_df.shape}")

# 3. Standardize โดย fit บน Train เท่านั้น แล้ว transform ทั้ง Train และ Test
scaler = StandardScaler()
X_train_std = scaler.fit_transform(train_df)
X_test_std = scaler.transform(test_df)

# 4. ทำ PCA โดย fit บน Train
pca = PCA()
pca.fit(X_train_std)

# 5. ฉายข้อมูล Test ลงบน PC1
test_scores = pca.transform(X_test_std)
test_pc1_scores = test_scores[:, 0]

# 6. หาความแปรปรวนตัวอย่าง (sample variance, ddof=1)
ans_410 = np.var(test_pc1_scores, ddof=1)
print(f"ความแปรปรวนตัวอย่างของคะแนน PC1 (ชุดทดสอบ): {ans_410:.4f}")
print(f"คำตอบข้อ 4.10: {ans_410:.4f}")


ขนาด Train: (960, 5), ขนาด Test: (243, 5)
ความแปรปรวนตัวอย่างของคะแนน PC1 (ชุดทดสอบ): 1.6239
คำตอบข้อ 4.10: 1.6239


## สรุปทฤษฎีและแนวคิดหลักประจำ Week 4

---

### 1. Eigenvalues และ Eigenvectors

สำหรับเมทริกซ์จัตุรัส $A \in \mathbb{R}^{n \times n}$ เวกเตอร์ $v \ne 0$ ที่เมื่อถูกแปลงเชิงเส้นด้วย $A$ แล้วมีทิศทางเดิมหรือตรงข้าม (เพียงถูกสเกลด้วยสเกลาร์ $\lambda$) เรียกว่า **eigenvector** และ $\lambda$ เรียกว่า **eigenvalue**:
$$Av = \lambda v \iff (A - \lambda I)v = 0$$

- การหา $\lambda$: หาจากสมการลักษณะเฉพาะ $\det(A - \lambda I) = 0$
- ใน NumPy ใช้ `np.linalg.eig(A)` ซึ่งจะคืนค่าทั้ง eigenvalues และ eigenvectors หรือ `np.linalg.eigvals(A)` เมื่อต้องการเฉพาะ eigenvalues
- การจัดเรียง: หากโจทย์ต้องการเรียงตามค่าพีชคณิต (algebraic value) จากมากไปน้อย ให้ใช้ `np.sort(np.real(eigvals))[::-1]` (ต่างจากการเรียงตามขนาดสัมบูรณ์ $|\lambda|$)

---

### 2. Singular Value Decomposition (SVD)

เมทริกซ์รูปสี่เหลี่ยมผืนผ้าใดๆ $A \in \mathbb{R}^{m \times n}$ สามารถแยกตัวประกอบได้เสมอเป็น:
$$A = U \Sigma V^T = \sum_{i=1}^{r} \sigma_i u_i v_i^T$$

โดยที่:
- $U \in \mathbb{R}^{m \times m}$ เป็น orthogonal matrix (คอลัมน์ $u_i$ คือ left singular vectors)
- $\Sigma \in \mathbb{R}^{m \times n}$ เป็น diagonal matrix บรรจุ singular values $\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_r \ge 0$
- $V \in \mathbb{R}^{n \times n}$ เป็น orthogonal matrix (คอลัมน์ $v_i$ คือ right singular vectors)
- ใน NumPy: `U, s, Vt = np.linalg.svd(A, full_matrices=False)` คืนค่า $s$ เป็น 1D array ของ singular values เรียงจากมากไปน้อย

---

### 3. Eckart-Young-Mirsky Theorem และ Low-Rank Approximation

ทฤษฎีบท Eckart-Young ระบุว่า การประมาณค่าเมทริกซ์ $A$ ด้วยเมทริกซ์ $A_k$ ที่มี rank ไม่เกิน $k$ ที่ดีที่สุดภายใต้ Frobenius norm หรือ Spectral norm คือการตัดตอนผลรวม SVD ให้เหลือเพียง $k$ เทอมแรก:
$$A_k = \sum_{i=1}^{k} \sigma_i u_i v_i^T = U_{:, :k} \Sigma_{:k, :k} V^T_{:k, :}$$

**คุณสมบัติสำคัญของความคลาดเคลื่อน (Frobenius Error):**
- ค่าความคลาดเคลื่อนสัมบูรณ์:
  $$\|A - A_k\|_F = \sqrt{\sum_{i=k+1}^{\min(m,n)} \sigma_i^2}$$
- ค่าความคลาดเคลื่อนสัมพัทธ์ (Relative Error):
  $$\frac{\|A - A_k\|_F}{\|A\|_F} = \frac{\sqrt{\sum_{i=k+1}^{\min(m,n)} \sigma_i^2}}{\sqrt{\sum_{i=1}^{\min(m,n)} \sigma_i^2}}$$

---

### 4. Principal Component Analysis (PCA)

PCA คือเทคนิคการลดมิติข้อมูลแบบเชิงเส้น โดยการหมุนพิกัดข้อมูลไปยังแกนใหม่ (Principal Axes) ที่เรียงตามทิศทางที่มีความแปรปรวน (Variance) มากที่สุดไปยังน้อยที่สุด

- **การหาแกน PC:** แกน $v_1$ (PC1) สอดคล้องกับ eigenvector ของ sample covariance matrix $S = \frac{1}{n-1} X_c^T X_c$ ที่คู่กับ eigenvalue ที่ใหญ่ที่สุด $\lambda_1$
- **ความสัมพันธ์กับ SVD:** เมื่อข้อมูลผ่านการ mean-center แล้ว $X_c = U \Sigma V^T$ จะได้ว่า right singular vectors $V$ ก็คือแกน PC ของข้อมูล และ $\lambda_i = \frac{\sigma_i^2}{n-1}$

---

### 5. Explained Variance และ Cumulative Variance Ratio

- ความแปรปรวนที่อธิบายได้โดยแกน PC ที่ $i$ คือ $\lambda_i$
- **สัดส่วนความแปรปรวน (Explained Variance Ratio):**
  $$\text{ratio}_i = \frac{\lambda_i}{\sum_{j=1}^{p} \lambda_j} = \frac{\sigma_i^2}{\sum_{j=1}^{p} \sigma_j^2}$$
- **สัดส่วนสะสม (Cumulative Explained Variance Ratio):**
  $$\text{cum\_ratio}_k = \sum_{i=1}^{k} \text{ratio}_i$$
  ใช้กำหนดจำนวน components ที่เหมาะสมในการเก็บข้อมูลให้อยู่ในระดับที่ต้องการ (เช่น $\ge 60\%$ หรือ $\ge 90\%$)

---

### 6. PCA Loadings และ Scores

- **Loading:** คือค่าน้ำหนักหรือสัมประสิทธิ์ใน eigenvector $v_i$ ที่มีความยาวหนึ่ง ($\sum v_{ji}^2 = 1$) บ่งบอกว่าตัวแปรดั้งเดิมมีอิทธิพลต่อ PC นั้นมากน้อยเพียงใด
- **PC Score:** คือพิกัดของข้อมูลจุดนั้นหลังฉายลงบนแกน PC:
  $$T = X_c V$$
  แถวที่ $i$ คอลัมน์ที่ $j$ ของ $T$ คือคะแนนบนแกน PC$j$ ของตัวอย่างข้อมูลที่ $i$
- **การใช้งานกับชุด Train / Test:**
  - ต้อง fit ค่าเฉลี่ย ส่วนเบี่ยงเบนมาตรฐาน และแกน PCA จากข้อมูลชุด Train เท่านั้น
  - นำพารามิเตอร์ดังกล่าวไป transform ชุด Test เพื่อป้องกัน Data Leakage